# 🔬 Tutorial SciML: Descubrimiento Inverso de Coeficiente de Arrastre ($C_d$) con PINNs
### **AWS Community Day Ecuador 2026**
**Ponente:** Jefferson Alfredo Conza Fajardo (*Universidad Yachay Tech* · *AWS Cloud Institute*)

---

## 📌 1. Planteamiento del Problema Inverso

En la dinámica clásica de caída libre con resistencia aerodinámica cuadrática, la ecuación de movimiento es:

$$m \frac{d^2 y}{d t^2} = -m g - \frac{1}{2} \rho A C_d \left( \frac{dy}{dt} \right) \left| \frac{dy}{dt} \right|$$

En el **problema inverso**, disponemos únicamente de observaciones ruidosas del sensor $\{(t_i, y_i)\}_{i=1}^N$ (altímetro o radar) y desconocemos el coeficiente de arrastre $C_d$.

La PINN parametriza la trayectoria $y_\theta(t)$ mediante una red neuronal y trata a $C_d$ como un **parámetro físico diferenciable** que se calibra automáticamente mediante gradientes con `torch.autograd`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Fijar semillas aleatorias para reproducibilidad
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Dispositivo de cómputo: {device}')

## 🛰️ 2. Generación de Datos Sintéticos con Ruido

In [ ]:
# Constantes físicas
m = 1.0       # Masa (kg)
g = 9.81      # Gravedad (m/s^2)
rho = 1.225   # Densidad del aire (kg/m^3)
A = 0.01      # Area frontal (m^2)
cd_true = 0.47 # Coeficiente Ground Truth (Esfera Lisa)
y0 = 100.0    # Altura inicial (m)

# Integrador RK4 analitico para generar datos sinteticos
t_eval = np.linspace(0, 3.0, 30)
y_curr = y0
v_curr = 0.0
y_exact = [y_curr]

dt = t_eval[1] - t_eval[0]
for _ in range(len(t_eval) - 1):
    # Substepping RK4
    def f(y, v):
        dv = -g - (0.5 * rho * A * cd_true / m) * v * abs(v)
        return v, dv
    
    k1_y, k1_v = f(y_curr, v_curr)
    k2_y, k2_v = f(y_curr + 0.5 * dt * k1_y, v_curr + 0.5 * dt * k1_v)
    k3_y, k3_v = f(y_curr + 0.5 * dt * k2_y, v_curr + 0.5 * dt * k2_v)
    k4_y, k4_v = f(y_curr + dt * k3_y, v_curr + dt * k3_v)
    
    y_curr += (dt / 6.0) * (k1_y + 2*k2_y + 2*k3_y + k4_y)
    v_curr += (dt / 6.0) * (k1_v + 2*k2_v + 2*k3_v + k4_v)
    y_exact.append(y_curr)

y_exact = np.array(y_exact)
noise_std = 0.05
y_sensor = y_exact + np.random.normal(0, noise_std, size=y_exact.shape)

print(f'Generados {len(t_eval)} puntos de sensor con ruido Gaussiano (std = {noise_std} m).')

## 🧠 3. Arquitectura PINN con Parámetro $C_d$ Entrenable

In [ ]:
class InverseDragPINN(nn.Module):
    def __init__(self, initial_cd_guess=0.10, y0=100.0, hidden_dim=64, num_layers=3):
        super().__init__()
        self.y0 = y0
        # Parametro entrenable Cd acotado positivamente
        self.raw_cd = nn.Parameter(torch.tensor(float(initial_cd_guess)).log())
        
        layers = [nn.Linear(1, hidden_dim), nn.Tanh()]
        for _ in range(num_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    @property
    def cd(self):
        return self.raw_cd.exp()

    def forward(self, t):
        # Garantiza condicion inicial exacta y(0) = y0
        return self.y0 + t * self.net(t)

model = InverseDragPINN(initial_cd_guess=0.10).to(device)
print(f'Suposicion inicial ciega de Cd: {model.cd.item():.4f} (Ground Truth: {cd_true})')

## ⚡ 4. Bucle de Calibración Inversa con Autograd

In [ ]:
t_sensor_t = torch.tensor(t_eval, dtype=torch.float32, device=device).view(-1, 1)
y_sensor_t = torch.tensor(y_sensor, dtype=torch.float32, device=device).view(-1, 1)
t_col = torch.linspace(0.0, 3.0, 50, device=device).view(-1, 1).requires_grad_(True)

optimizer = optim.Adam([
    {'params': model.net.parameters(), 'lr': 1e-2},
    {'params': [model.raw_cd], 'lr': 5e-2},
])

epochs = 300
cd_history = []

print(f'🔥 Calibrando PINN Inversa ({epochs} épocas)...')
for epoch in range(1, epochs + 1):
    optimizer.zero_grad()

    # 1. Perdida de datos del sensor
    y_pred = model(t_sensor_t)
    loss_data = torch.mean((y_pred - y_sensor_t) ** 2)

    # 2. Perdida residual de la ODE (Newton)
    y_col = model(t_col)
    v_col = torch.autograd.grad(y_col, t_col, grad_outputs=torch.ones_like(y_col), create_graph=True)[0]
    a_col = torch.autograd.grad(v_col, t_col, grad_outputs=torch.ones_like(v_col), create_graph=True)[0]

    ode_res = m * a_col - (-m * g - 0.5 * rho * A * model.cd * v_col * torch.abs(v_col))
    loss_ode = torch.mean(ode_res ** 2)

    loss = loss_data + 20.0 * loss_ode
    loss.backward()
    optimizer.step()

    cd_val = model.cd.item()
    cd_history.append(cd_val)

    if epoch % 50 == 0 or epoch == 1:
        rel_err = abs(cd_val - cd_true) / cd_true * 100
        print(f'[{epoch:03d}/{epochs}] Cd: {cd_val:.4f} | Error: {rel_err:.2f}% | Loss Total: {loss.item():.5f}')

## 📊 5. Visualización del Descubrimiento y Ajuste de Trayectoria

In [ ]:
t_dense = torch.linspace(0, 3, 200, device=device).view(-1, 1)
with torch.no_grad():
    y_dense_pred = model(t_dense).cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Grafica 1: Trayectoria predicha vs mediciones ruidosas
ax1.scatter(t_eval, y_sensor, color='crimson', s=40, label='Sensor Radar (Ruido)')
ax1.plot(t_eval, y_exact, 'k--', label=f'Ground Truth Exacto (Cd={cd_true})')
ax1.plot(t_dense.cpu().numpy(), y_dense_pred, color='#FF9900', lw=2.5, label=f'PINN Ajuste (Cd Descubierto={model.cd.item():.4f})')
ax1.set_title('Trayectoria Dinámica y(t)')
ax1.set_xlabel('Tiempo (s)')
ax1.set_ylabel('Altitud (m)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Grafica 2: Convergencia de Cd
ax2.plot(cd_history, color='#00A4E4', lw=2, label='Evolución Cd Estimado')
ax2.axhline(cd_true, color='red', linestyle='--', label=f'Cd Real ({cd_true})')
ax2.set_title('Descubrimiento del Parámetro Físico Cd')
ax2.set_xlabel('Época')
ax2.set_ylabel('Valor de Cd')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()